# E06: Libreria Estandar Avanzada de Python

## Objetivos de aprendizaje

Al finalizar esta sesion, seras capaz de:

1. Utilizar `collections` (`deque`, `defaultdict`, `Counter`, `namedtuple`, `ChainMap`) para resolver problemas comunes de conteo, agrupacion y almacenamiento de datos con codigo mas expressivo.
2. Combinar iteradores avanzados de `itertools` (`chain`, `product`, `groupby`, `accumulate`, `islice`, `takewhile`) para crear pipelines de datos eficientes sin listas intermedias.
3. Aplicar `functools` (`reduce`, `partial`, `lru_cache`, `singledispatch`, `wraps`, `total_ordering`) para escribir codigo funcional, reutilizable y con cache.
4. Dominar `pathlib` avanzado para navegar, filtrar y manipular archivos del sistema de forma orientada a objetos.
5. Configurar y usar `logging` profesional con niveles, handlers y formateo personalizado, eliminando `print` como herramienta de diagnostico.
6. Manejar fechas y horas con `datetime` moderno: tipos aware vs naive, timedelta, strftime/strptime y timezone.

## Analogia: La biblioteca de referencia

La libreria estandar de Python es como la **biblioteca de referencia** de una universidad: esta preinstalada en el edificio (la instalacion de Python), no necesitas internet (instalar paquetes externos) ni una tarjeta de credito (pagar licencias). Solo necesitas **saber donde buscar**.

```
+-------------------------------------------------------------+
|           BIBLIOTECA DE REFERENCIA  (stdlib)                |
|                                                             |
|  +-------------------+  +-----------------+  +------------+ |
|  | collections       |  | itertools       |  | functools  | |
|  | (organizar datos) |  | (iterar datos)  |  | (funciones)| |
|  +-------------------+  +-----------------+  +------------+ |
|  +-------------------+  +-----------------+  +------------+ |
|  | pathlib           |  | logging         |  | datetime   | |
|  | (archivos)        |  | (diagnostico)   |  | (tiempo)   | |
|  +-------------------+  +-----------------+  +------------+ |
+-------------------------------------------------------------+

  La mayoria de los programadores novatos solo usan print() y pip install.
  El programador avanzado resuelve el 80% de sus problemas
  SIN instalar NINGUN paquete externo.
```

> **Filosofia**: Antes de instalar un paquete externo, pregunta: "Esto ya viene con Python?". La respuesta mas frecuente es si.

## 1. collections - Contenedores especializados

El modulo `collections` extiende las estructuras basicas de Python (`list`, `dict`, `tuple`) con variantes mas potentes para escenarios especificos.

### 1.1 deque - Cola de doble extremo

Un `deque` (double-ended queue) permite agregar y eliminar elementos por **ambos extremos** en O(1). A diferencia de una lista, donde `list.pop(0)` es O(n) porque mueve todos los elementos, el deque es ideal para colas y ventanas deslizantes.

In [ ]:
from collections import deque

dq = deque([1, 2, 3])

# Agregar/eliminar por ambos extremos
dq.appendleft(0)   # por la izquierda
dq.append(4)       # por la derecha
print(f"Despues de agregar: {dq}")

dq.popleft()       # eliminar izquierda
dq.pop()           # eliminar derecha
print(f"Despues de eliminar: {dq}")

# Rotar elementos
dq2 = deque([1, 2, 3, 4, 5])
dq2.rotate(2)  # mueve los ultimos 2 al frente
print(f"Rotado a la derecha: {dq2}")

dq2.rotate(-1) # mueve el primero al final
print(f"Rotado a la izquierda: {dq2}")

# Ventana deslizante (sliding window)
datos = [10, 20, 30, 40, 50]
ventana = deque(maxlen=3)  # limite automatico
for valor in datos:
    ventana.append(valor)
    print(f"  Ventana: {list(ventana)} -> promedio: {sum(ventana)/len(ventana):.1f}")

### 1.2 defaultdict - Diccionario con valores por defecto

Un `defaultdict` elimina la necesidad de verificar si una clave existe (`if key in dict`) o usar `dict.setdefault()`. Si accedes a una clave inexistente, crea automaticamente el valor usando la **funcion factory** que le pasaste.

In [ ]:
from collections import defaultdict

# Agrupar palabras por primera letra
palabras = ["manzana", "mango", "banana", "arandano", "cereza"]
por_letra = defaultdict(list)

for palabra in palabras:
    por_letra[palabra[0]].append(palabra)  # nunca KeyError

print(dict(por_letra))

# Contar frecuencias con int factory
frutas = ["manzana", "banana", "manzana", "cereza", "banana", "manzana"]
conteo = defaultdict(int)
for fruta in frutas:
    conteo[fruta] += 1  # defaultdict(int) empieza en 0

print(dict(conteo))

# defaultdict anidado (dict de dicts de listas)
datos = defaultdict(lambda: defaultdict(list))
datos["ventas"]["2024"].append(1000)
datos["ventas"]["2024"].append(2000)
datos["ventas"]["2025"].append(1500)
print(dict({k: dict(v) for k, v in datos.items()}))

### 1.3 Counter - Contador de elementos

Un `Counter` es un diccionario optimizado para contar elementos. Tiene metodos como `most_common()`, operaciones aritmeticas entre counters, y puede crearse directamente desde un iterable.

In [ ]:
from collections import Counter

# Contar caracteres en un texto
texto = "python es genial"
letras = Counter(texto)
print(f"Letras mas comunes: {letras.most_common(3)}")
print(f"Total caracteres: {sum(letras.values())}")

# Contar palabras en un parrafo
parrafo = "el gato dormia en el sofa y el sofa estaba comodo"
palabras = parrafo.split()
conteo_palabras = Counter(palabras)
print(f"\nPalabras: {conteo_palabras}")
print(f"Top 2: {conteo_palabras.most_common(2)}")

# Aritmetica con Counter
semestre_1 = Counter({"python": 5, "sql": 3, "excel": 2})
semestre_2 = Counter({"python": 4, "tableau": 3, "sql": 5})

total = semestre_1 + semestre_2           # suma de conteos
diferencia = semestre_1 - semestre_2      # solo positivos
comun = semestre_1 & semestre_2           # interseccion (min)

print(f"\nTotal semestres: {dict(total)}")
print(f"Solo semestre 1 (no compartido): {dict(diferencia)}")
print(f"Comun (min): {dict(comun)}")

### 1.4 namedtuple y OrderedDict

In [ ]:
from collections import namedtuple, OrderedDict

# namedtuple - tupla con nombre accedible por campo
Punto = namedtuple("Punto", ["x", "y"])
p = Punto(3, 4)
print(f"Punto: {p}")
print(f"x={p.x}, y={p.y}")
print(f"Como tupla: {tuple(p)}")

# OrderedDict - diccionario que respeta orden de insercion
# (En Python 3.7+ dict normal ya preserva orden, pero OrderedDict
# agrega popitem(last=True) y move_to_end())
od = OrderedDict()
od["a"] = 1
od["b"] = 2
od["c"] = 3
od.move_to_end("a")
print(f"\nDespues de mover 'a' al final: {dict(od)}")
od.popitem(last=False)
print(f"Despues de popitem del primero: {dict(od)}")

### 1.5 ChainMap - Mapeo encadenado

Un `ChainMap` combina varios diccionarios en una **vista unificada**. Al buscar una clave, recorre los diccionarios en orden: el primero que la contiene la gana. Es util para configuraciones con defaults y overrides.

In [ ]:
from collections import ChainMap

# Configuracion por defecto + overrides del usuario
defaults = {"color": "azul", "tamaño": "mediano", "idioma": "es"}
usuario = {"color": "rojo", "idioma": "en"}

config = ChainMap(usuario, defaults)  # usuario tiene prioridad

print(f"color: {config['color']}")       # de usuario
print(f"tamaño: {config['tamaño']}")     # de defaults
print(f"idioma: {config['idioma']}")     # de usuario
print(f"\nTodos los valores: {dict(config)}")

# new_child() crea una capa temporal encima
temp = config.new_child({"color": "verde"})
print(f"Temp override: {temp['color']}")
print(f"Original: {config['color']}")

## 2. itertools - Iteradores de alto rendimiento

El modulo `itertools` proporciona **funciones iterator** que trabajan en conjunto con la iteracion de Python. Son ideales para pipelines de datos: transforman, combinan y filtran secuencias de forma **lazy** (evaluacion perezosa), sin crear listas intermedias en memoria.

### 2.1 chain, cycle, repeat, islice

In [ ]:
from itertools import chain, cycle, repeat, islice

# chain: concatenar iterables sin crear una lista nueva
lista_a = [1, 2, 3]
lista_b = [4, 5, 6]
concatenado = list(chain(lista_a, lista_b))
print(f"chain: {concatenado}")

# chain.from_iterable: encadenar N iterables
matrices = [[1, 2], [3, 4], [5, 6]]
plano = list(chain.from_iterable(matrices))
print(f"Aplanar: {plano}")

# cycle: repetir un iterable infinitamente
ciclo = list(islice(cycle(["A", "B", "C"]), 8))
print(f"cycle limitado: {ciclo}")

# repeat: repetir un valor
reps = list(islice(repeat(42, 5), 8))  # 42, 42, 42, 42, 42, None, None, None
print(f"repeat: {reps}")

# islice: rebanar iteradores (como list slicing pero lazy)
numeros = range(20)
rebanado = list(islice(numeros, 5, 15, 2))  # start, stop, step
print(f"islice(5,15,2): {rebanado}")

### 2.2 takewhile y dropwhile

In [ ]:
from itertools import takewhile, dropwhile

# takewhile: tomar MIENTRAS la condicion sea True
numeros = [1, 3, 5, 8, 2, 4, 6]
pares_alcansados = list(takewhile(lambda x: x % 2 != 0, numeros))
print(f"takewhile (impares): {pares_alcansados}")

# dropwhile: saltar MIENTRAS la condicion sea True, luego tomar todo
despues_de_impares = list(dropwhile(lambda x: x % 2 != 0, numeros))
print(f"dropwhile (impares): {despues_de_impares}")

# Ejemplo practical: leer registros hasta encontrar uno invalido
registros = [10, 20, 30, 0, 40, 50]
validos = list(takewhile(lambda x: x > 0, registros))
print(f"\nRegistros validos: {validos}")

### 2.3 product, permutations, combinations

Estas funciones generan **todas las combinaciones posibles** de elementos. Son esenciales en analisis de datos para generar espacio de busqueda o pruebas exhaustivas.

In [ ]:
from itertools import product, permutations, combinations, combinations_with_replacement

# product: producto cartesiano (todos contra todos)
colores = ["rojo", "azul"]
tamaños = ["S", "M", "L"]
productos = list(product(colores, tamaños))
print(f"Producto cartesiano: {len(productos)} combinaciones")
print(f"Ejemplo: {productos[:4]}")

# permutations: permutaciones (orden importa)
elems = ["A", "B", "C"]
perm_2 = list(permutations(elems, 2))
print(f"\nPermutaciones(3,2): {perm_2}")
print(f"Total: {len(perm_2)}")

# combinations: combinaciones (orden NO importa)
comb_2 = list(combinations(elems, 2))
print(f"\nCombinaciones(3,2): {comb_2}")
print(f"Total: {len(comb_2)}")

# combinations_with_replacement: permite repetir elementos
comb_rep = list(combinations_with_replacement([0, 1], 3))
print(f"\nComb con repetición(0,1, 3): {comb_rep}")

### 2.4 groupby, starmap, accumulate

In [ ]:
from itertools import groupby, starmap, accumulate
from operator import mul

# groupby: agrupar elementos CONSECUTIVOS con la misma clave
# IMPORTANTE: los datos DEBEN estar ordenados por la clave
ventas = [
    ("2024-Q1", 100), ("2024-Q1", 150), ("2024-Q2", 200),
    ("2024-Q2", 180), ("2025-Q1", 220), ("2025-Q1", 190)
]
ventas.sort(key=lambda x: x[0])  # ordenar por trimestre

for clave, grupo in groupby(ventas, key=lambda x: x[0]):
    items = list(grupo)
    total = sum(monto for _, monto in items)
    print(f"  {clave}: {len(items)} registros, total = {total}")

# accumulate: acumulador running (como reduce pero muestra pasos)
numeros = [1, 2, 3, 4, 5]
acum_suma = list(accumulate(numeros))
print(f"\nAcumulado (suma): {acum_suma}")

acum_mult = list(accumulate(numeros, mul))
print(f"Acumulado (producto): {acum_mult}")

# starmap: map que desempaqueta tuplas como argumentos
pares = [(2, 3), (4, 5), (1, 6)]
potencias = list(starmap(pow, pares))
print(f"\nstarmap(pow, pares): {potencias}")  # 2^3, 4^5, 1^6

## 3. functools - Funciones de orden superior

El modulo `functools` contiene herramientas para **tratar funciones como objetos de primera clase**: almacenar resultados, pre-configurar argumentos, crear dispatchers por tipo y ordenar comparaciones.

### 3.1 reduce

Aplica una funcion binaria acumulativamente sobre un iterable, reduciendolo a un solo valor.

In [ ]:
from functools import reduce
from operator import add, mul

# Suma acumulada (equivalente a sum())
numeros = [1, 2, 3, 4, 5]
suma = reduce(add, numeros)
print(f"reduce(add, {numeros}) = {suma}")

# Producto (no hay built-in para esto)
producto = reduce(mul, numeros)
print(f"reduce(mul, {numeros}) = {producto}")

# Encontrar el maximo manualmente
maximo = reduce(lambda a, b: a if a > b else b, numeros)
print(f"reduce(max, {numeros}) = {maximo}")

# Aplanar una lista de listas
listas = [[1, 2], [3, 4], [5, 6]]
plano = reduce(lambda a, b: a + b, listas)
print(f"Aplanar: {plano}")

### 3.2 partial - Funciones pre-configuradas

Un `partial` crea una **nueva funcion** con algunos argumentos ya fijados. Es util para crear funciones especializadas a partir de funciones generales.

In [ ]:
from functools import partial

# Funcion general
def potencia(base, exponente):
    return base ** exponente

# Funciones especializadas con partial
cuadrado = partial(potencia, exponente=2)
cubo = partial(potencia, exponente=3)

print(f"cuadrado(5) = {cuadrado(5)}")
print(f"cubo(3) = {cubo(3)}")

# Practico: filtrar con predicados pre-configurados
def filtrar_por_campo(registros, campo, valor):
    return [r for r in registros if r.get(campo) == valor]

ventas = [
    {"producto": "A", "region": "Norte", "monto": 100},
    {"producto": "B", "region": "Sur", "monto": 200},
    {"producto": "C", "region": "Norte", "monto": 150},
]

filtrar_norte = partial(filtrar_por_campo, campo="region", valor="Norte")
print(f"\nNorte: {filtrar_norte(ventas)}")

### 3.3 lru_cache - Memoizacion con cache

Un `lru_cache` almacena los resultados de una funcion para que no se recalcule con los mismos argumentos. Es especialmente poderoso en **recursion** y **llamadas repetidas** con los mismos datos.

In [ ]:
from functools import lru_cache
import time

@lru_cache(maxsize=128)
def fibonacci(n: int) -> int:
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

# Sin cache: O(2^n) | Con cache: O(n)
inicio = time.perf_counter()
resultado = fibonacci(40)
transcurrido = time.perf_counter() - inicio
print(f"fibonacci(40) = {resultado}")
print(f"Tiempo: {transcurrido:.4f}s")
print(f"Llamadas cacheadas: {fibonacci.cache_info()}")

# Limpiar cache si ya no se necesita
fibonacci.cache_clear()

### 3.4 singledispatch - Dispatch por tipo

`singledispatch` permite definir una funcion que se comporta diferente segun el **tipo** de su primer argumento. Es el equivalente a la sobrecarga de metodos en otros lenguajes.

In [ ]:
from functools import singledispatch

@singledispatch
def procesar_dato(dato):
    raise TypeError(f"Tipo no soportado: {type(dato)}")

@procesar_dato.register
def _(dato: str):
    return f"Texto en mayusculas: {dato.upper()}"

@procesar_dato.register
def _(dato: int):
    return f"Numero al cuadrado: {dato ** 2}"

@procesar_dato.register
def _(dato: list):
    return f"Lista con {len(dato)} elementos, suma = {sum(dato)}"

# Dispatch automatico por tipo
print(procesar_dato("hola mundo"))
print(procesar_dato(7))
print(procesar_dato([10, 20, 30]))

### 3.5 wraps y total_ordering

In [ ]:
from functools import wraps, total_ordering
import time
import logging

# wraps: preserva metadatos de la funcion original al decorar
def mi_timer(func):
    @wraps(func)  # sin esto, la funcion pierde __name__ y __doc__
    def wrapper(*args, **kwargs):
        inicio = time.perf_counter()
        resultado = func(*args, **kwargs)
        transcurrido = time.perf_counter() - inicio
        print(f"{func.__name__} ejecutada en {transcurrido:.6f}s")
        return resultado
    return wrapper

@mi_timer
def sumar(a, b):
    """Suma dos numeros."""
    return a + b

print(f"Nombre preservado: {sumar.__name__}")
print(f"Docstring preservado: {sumar.__doc__}")
print(f"Resultado: {sumar(3, 4)}")

# total_ordering: genera metodos de comparacion faltantes
@total_ordering
class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)
    def __lt__(self, other):  # solo define esto
        return (self.x, self.y) < (other.x, other.y)
    # total_ordering genera automaticamente: __le__, __gt__, __ge__

p1, p2, p3 = Punto(1, 2), Punto(1, 2), Punto(3, 4)
print(f"\np1 == p2: {p1 == p2}")
print(f"p1 < p3: {p1 < p3}")
print(f"p1 >= p3: {p1 >= p3}")  # generado por total_ordering

## 4. pathlib avanzado

`pathlib` es la forma moderna y orientada a objetos de trabajar con rutas del sistema de archivos. Reemplaza las cadenas de texto crudas y `os.path` con objetos `Path` que se manipulan con metodos claros.

In [ ]:
from pathlib import Path
import os

# Crear un Path de prueba
base = Path.cwd()  # directorio actual
print(f"Directorio actual: {base}")
print(f"Nombre: {base.name}")
print(f"Extension: {base.suffix}")
print(f"Sin extension: {base.stem}")
print(f"Padre: {base.parent}")

# Construir rutas con / ( sobrecarga del operador )
ruta_archivo = base / "datos" / "ventas.csv"
print(f"\nRuta construida: {ruta_archivo}")
print(f"Existe: {ruta_archivo.exists()}")

### 4.1 glob y rglob - Busqueda de archivos

In [ ]:
from pathlib import Path

# Buscar notebooks en el directorio actual (un nivel)
directorio = Path.cwd()
notebooks = list(directorio.glob("*.ipynb"))
print(f"Notebooks en directorio actual: {len(notebooks)}")
for nb in notebooks[:3]:  # mostrar solo los primeros 3
    print(f"  - {nb.name}")

# rglob: busqueda recursiva
todos_notebooks = list(directorio.parent.rglob("*.ipynb"))
print(f"\nNotebooks recursivos: {len(todos_notebooks)}")

### 4.2 Lectura y escritura de archivos

In [ ]:
from pathlib import Path
import json

# Crear archivo de prueba
archivo_test = Path("test_ejemplo.txt")
archivo_test.write_text("Hola desde pathlib\nLinea 2\nLinea 3", encoding="utf-8")
print(f"Leido: {archivo_test.read_text(encoding='utf-8')}")

# JSON con pathlib
datos = {"nombre": "Python", "version": 3.12, "modulos": 400}
archivo_json = Path("test_datos.json")
archivo_json.write_text(json.dumps(datos, indent=2, ensure_ascii=False), encoding="utf-8")
contenido = json.loads(archivo_json.read_text(encoding="utf-8"))
print(f"\nJSON leido: {contenido}")

# Modificar sufijo y nombre
original = Path("informe_2024.csv")
print(f"\nOriginal: {original}")
print(f"Con sufijo .xlsx: {original.with_suffix('.xlsx')}")
print(f"Con sufijo .bak: {original.with_suffix('.bak')}")

# Cleanup
archivo_test.unlink(missing_ok=True)
archivo_json.unlink(missing_ok=True)

### 4.3 relative_to y timestamp

In [ ]:
from pathlib import Path
import datetime

# relative_to: ruta relativa a un directorio base
absoluta = Path("C:/Users/luisj/Github/Datajupyer/consolidado/intermedio/E06.ipynb")
try:
    relativa = absoluta.relative_to("C:/Users/luisj/Github/Datajupyer")
    print(f"Relativa: {relativa}")
except ValueError as e:
    print(f"Error: {e}")

# Timestamps de archivos existentes
archivo = Path.cwd() / "E06_libreria_estandar.ipynb"
if archivo.exists():
    stat = archivo.stat()
    creado = datetime.datetime.fromtimestamp(stat.st_ctime)
    modificado = datetime.datetime.fromtimestamp(stat.st_mtime)
    print(f"\nArchivo: {archivo.name}")
    print(f"  Creado: {creado}")
    print(f"  Modificado: {modificado}")
    print(f"  Tamano: {stat.st_size:,} bytes")
else:
    print("Archivo no encontrado (se creara al ejecutar el notebook)")

## 5. logging profesional

En produccion, **`print()` no sirve para diagnostico**: no tiene niveles, no se puede desactivar, no se puede redirigir a un archivo, y no tiene timestamps. `logging` es la solucion profesional para todas estas limitaciones.

### 5.1 Niveles de logging

```
NIVEL         | VALOR | CUANDO USARLO
──────────────┼───────┼─────────────────────────────────────
DEBUG         |  10   | Variables internas, flujo detallado
INFO          |  20   | Mensajes de confirmacion normal
WARNING       |  30   | Algo inesperado pero recuperable
ERROR         |  40   | Algo fallo, la funcion no pudo completar
CRITICAL      |  50   | El sistema entero esta en peligro
```

La regla de oro: **print() para mostrar resultados al usuario, logging() para diagnosticar el programa**.

In [ ]:
import logging

# Configuracion basica: nivel minimo y formato
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logger = logging.getLogger(__name__)

# Los niveles se filtran automaticamente
logger.debug("Variable x = 42")
logger.info("Proceso iniciado")
logger.warning("Memoria al 85%%")
logger.error("Conexion a BD fallida")
# logger.critical("Sistema caido")  # no ejecutar en ejemplo

### 5.2 Handlers: Stream + File

Un **handler** define **donde** se envian los logs. Puedes tener multiples handlers: consola (Stream) para ver en vivo, y archivo (File) para persistir.

In [ ]:
import logging
from pathlib import Path

# Crear logger con nombre especifico
logger_avanzado = logging.getLogger("app.ventas")
logger_avanzado.setLevel(logging.DEBUG)

# Handler de consola: solo WARNING en adelante
consola = logging.StreamHandler()
consola.setLevel(logging.WARNING)
consola.setFormatter(logging.Formatter(
    "%(levelname)-8s | %(message)s"
))

archivo = logging.FileHandler("app.log", encoding="utf-8")
archivo.setLevel(logging.DEBUG)  # todo se guarda en archivo
archivo.setFormatter(logging.Formatter(
    "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S"
))

logger_avanzado.addHandler(consola)
logger_avanzado.addHandler(archivo)

# Estos van al archivo pero NO a consola (DEBUG < WARNING)
logger_avanzado.debug("Leyendo base de datos...")
logger_avanzado.info("100 registros procesados")

# Estos van AMBOS: consola y archivo
logger_avanzado.warning("Timeout en conexion externa")
logger_avanzado.error("No se pudo escribir archivo de salida")

# Leer el archivo generado
from pathlib import Path as P
log_file = P("app.log")
if log_file.exists():
    print("\n--- Contenido del log ---")
    print(log_file.read_text(encoding="utf-8"))
    log_file.unlink()  # limpiar

### 5.3 logging en un script real

In [ ]:
import logging

# Patron tipico: configurar UNA VEZ al inicio del script
def configurar_logging(nivel: str = "INFO", archivo: str | None = None) -> None:
    """Configura el sistema de logging del programa."""
    nivel_num = getattr(logging, nivel.upper(), logging.INFO)

    handlers: list[logging.Handler] = [logging.StreamHandler()]
    if archivo:
        handlers.append(logging.FileHandler(archivo, encoding="utf-8"))

    logging.basicConfig(
        level=nivel_num,
        format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        handlers=handlers,
        force=True,  # sobreescribe configuracion previa
    )

# Simular un script de procesamiento de datos
configurar_logging("DEBUG")
logger = logging.getLogger("procesador")

datos_crudos = [10, 20, None, 40, None, 60]
logger.info(f"Recibidos {len(datos_crudos)} registros")

limpios = []
for i, valor in enumerate(datos_crudos):
    if valor is None:
        logger.warning(f"Registro {i}: valor nulo, omitiendo")
        continue
    limpios.append(valor)
    logger.debug(f"Registro {i}: {valor} procesado")

promedio = sum(limpios) / len(limpios) if limpios else 0
logger.info(f"Resultado: {len(limpios)} validos, promedio = {promedio:.1f}")
logger.debug("Procesamiento completado")

## 6. datetime moderno

El modulo `datetime` gestiona fechas, horas y duraciones. Python 3.12+ moderniza su uso con type hints mas claros y soporte nativo para timezone-aware.

### 6.1 Jerarquia de tipos

```
                    datetime
                   (fecha + hora)
                    /         \
                date           time
            (solo fecha)    (solo hora)

+-------------+     +-------------+
|  timedelta  |     |  timezone   |
| (duracion)  |     | (UTC offset)|
+-------------+     +-------------+

Relaciones clave:
  date + timedelta -> date
  datetime + timedelta -> datetime
  datetime - datetime -> timedelta
  datetime + timezone -> aware datetime
```

### 6.2 Aware vs Naive

- **Naive**: datetime sin zona horaria (no sabe si es UTC, local, etc). Prone a bugs en aplicaciones globales.
- **Aware**: datetime con zona horaria. Siempre preferir en produccion.

```
  NAIVE:  datetime(2024, 6, 15, 14, 30)         # ambiguous
  AWARE:  datetime(2024, 6, 15, 14, 30, tzinfo=UTC)  # preciso
```

In [ ]:
from datetime import date, time, datetime, timedelta, timezone

# date: solo fecha
hoy = date.today()
print(f"Hoy: {hoy}")
print(f"  Year: {hoy.year}, Mes: {hoy.month}, Dia: {hoy.day}")
print(f"  Dia de la semana: {hoy.strftime('%A')}")
print(f"  Es bisiesto: {hoy.year % 4 == 0 and (hoy.year % 100 != 0 or hoy.year % 400 == 0)}")

# time: solo hora
ahora = time(14, 30, 45)
print(f"\nHora: {ahora}")

# datetime: fecha + hora (naive por defecto)
dt_naive = datetime(2024, 6, 15, 14, 30)
dt_aware = datetime(2024, 6, 15, 14, 30, tzinfo=timezone.utc)
print(f"\nNaive: {dt_naive}")
print(f"Aware: {dt_aware}")
print(f"Tiene zona horaria: {dt_aware.tzinfo is not None}")

# datetime.now() con timezone
utc_now = datetime.now(timezone.utc)
print(f"\nUTC ahora: {utc_now}")

### 6.3 timedelta - Diferencias y aritmetica

In [ ]:
from datetime import datetime, timedelta, timezone

# Diferencia entre dos fechas
inicio = datetime(2024, 1, 1)
fin = datetime(2024, 12, 31)
diferencia = fin - inicio

print(f"Dias en 2024: {diferencia.days}")
print(f"Segundos totales: {diferencia.total_seconds():,.0f}")

# Aritmetica con timedelta
hoy = datetime.now(timezone.utc)
manana = hoy + timedelta(days=1)
en_una_semana = hoy + timedelta(weeks=1)
en_3_horas = hoy + timedelta(hours=3)

print(f"\nHoy:      {hoy.strftime('%Y-%m-%d %H:%M')}")
print(f"Manana:   {manana.strftime('%Y-%m-%d %H:%M')}")
print(f"En 1 sem: {en_una_semana.strftime('%Y-%m-%d %H:%M')}")
print(f"En 3 hrs: {en_3_horas.strftime('%Y-%m-%d %H:%M')}")

### 6.4 strftime y strptime - Formateo

In [ ]:
from datetime import datetime

# strftime: datetime -> string
ahora = datetime(2024, 7, 15, 14, 30, 0)

formatos = {
    "ISO 8601":      ahora.isoformat(),
    "DD/MM/AAAA":    ahora.strftime("%d/%m/%Y"),
    "Nombre dia":    ahora.strftime("%A %d de %B %Y"),
    "Timestamp":     ahora.strftime("%Y%m%d_%H%M%S"),
    "Corto":         ahora.strftime("%d-%b-%y"),
}

for nombre, valor in formatos.items():
    print(f"  {nombre:15}: {valor}")

# strptime: string -> datetime
print("\nParseando strings:")
cadenas = [
    ("2024-07-15", "%Y-%m-%d"),
    ("15/07/2024", "%d/%m/%Y"),
    ("Jul 15, 2024", "%b %d, %Y"),
]
for cadena, fmt in cadenas:
    fecha = datetime.strptime(cadena, fmt)
    print(f"  '{cadena}' -> {fecha}")

### 6.5 datetime: Diagrama de relaciones

```
                      +------------------+
                      |   timedelta      |
                      |  (duracion)      |
                      +--------+---------+
                               |
              resta de         |   suma con
          +--------------------+------------------+
          |                                       |
   +------+------+                     +----------+----------+
   |    date     |                     |     datetime         |
   | (ano,mes,d) |                     | (ano,mes,d,h,m,s)   |
   +------+------+                     +----------+----------+
          |                                       |
          +--------  date + timedelta -----------+
                                                      |
                                          +-----------+-----------+
                                          |  timezone.utc         |
                                          |  (zona horaria)       |
                                          +-----------------------+

   Naive datetime  + timezone  -->  Aware datetime
   (peligroso)     (UTC/Offset)    (seguro)

   strftime:  datetime  -->  string  (para mostrar)
   strptime:  string    -->  datetime (para parsear)
```

## 7. Otras utilidades clave

### 7.1 pprint - Pretty print

Imprime estructuras de datos de forma legible, con sangria y niveles.

In [ ]:
from pprint import pprint, pformat

datos_empleados = [
    {"nombre": "Ana", "departamento": "Data", "habilidades": ["Python", "SQL", "Spark"]},
    {"nombre": "Luis", "departamento": "Eng", "habilidades": ["Go", "K8s", "Docker"]},
    {"nombre": "Mia", "departamento": "Data", "habilidades": ["R", "TensorFlow"]},
]

print("--- print normal (ilegible) ---")
print(str(datos_empleados))

print("\n--- pprint (legible) ---")
pprint(datos_empleados, width=60, depth=None)

# pformat: retorna string en lugar de imprimir
formateado = pformat(datos_empleados, indent=2)
print(f"\n--- pformat como string (primeros 200 chars) ---")
print(formateado[:200] + "...")

### 7.2 json avanzado

In [ ]:
import json
from datetime import datetime, date
from pathlib import Path

# Encoder personalizado: serializar tipos que json no soporta
class DateTimeEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, datetime):
            return o.isoformat()
        if isinstance(o, date):
            return o.isoformat()
        return super().default(o)

registro = {
    "evento": "venta",
    "monto": 1500.50,
    "fecha": datetime.now(),
    "items": ["A", "B", "C"],
    "metadata": None,
}

# Dumps con encoder y formato legible
json_str = json.dumps(
    registro,
    cls=DateTimeEncoder,
    indent=2,
    ensure_ascii=False,
    sort_keys=False,
    default=str,  # fallback para tipos no serializables
)
print(json_str)

# Carga con object_hook (para decodificar tipos personalizados)
cargado = json.loads(json_str)
print(f"\nTipo fecha despues de carga: {type(cargado['fecha'])}")

### 7.3 math y statistics

In [ ]:
import math
import statistics

# math: constantes y funciones matematicas
print("--- math ---")
print(f"pi: {math.pi:.6f}")
print(f"e: {math.e:.6f}")
print(f"ceil(3.2): {math.ceil(3.2)}")
print(f"floor(3.8): {math.floor(3.8)}")
print(f"sqrt(144): {math.sqrt(144)}")
print(f"log(1000, 10): {math.log(1000, 10)}")
print(f"distancia (3,4): {math.dist((0,0), (3,4))}")

# statistics: estadistica descriptiva
datos = [23, 45, 12, 67, 34, 89, 23, 56, 78, 45]
print(f"\n--- statistics ---")
print(f"Media: {statistics.mean(datos)}")
print(f"Mediana: {statistics.median(datos)}")
print(f"Moda: {statistics.mode(datos)}")
print(f"Desviacion estandar: {statistics.stdev(datos):.2f}")
print(f"Varianza: {statistics.variance(datos):.2f}")
print(f"Cuantiles (Q1, Q2, Q3): {statistics.quantiles(datos, n=4)}")

### 7.4 timeit - Medir rendimiento

In [ ]:
import timeit

# Comparar rendimiento de dos implementaciones
setup = "data = list(range(1000))"

tiempo_sum = timeit.timeit(
    stmt="s = 0\nfor x in data: s += x",
    setup=setup,
    number=10000
)

tiempo_sum_builtin = timeit.timeit(
    stmt="s = sum(data)",
    setup=setup,
    number=10000
)

print(f"Bucle manual:  {tiempo_sum:.4f}s")
print(f"sum() built-in: {tiempo_sum_builtin:.4f}s")
print(f"sum() es {tiempo_sum / tiempo_sum_builtin:.1f}x mas rapido")

# timeit desde la linea de comandos:
# python -m timeit "sum(range(1000))"

## 8. Tabla de referencia - Resumen de la stdlib

| Modulo | Clases/Funciones clave | Uso principal |
|--------|----------------------|---------------|
| `collections.deque` | `deque`, `appendleft`, `popleft`, `rotate` | Colas, ventanas deslizantes, O(1) en ambos extremos |
| `collections.defaultdict` | `defaultdict(list)`, `defaultdict(int)` | Eliminar `if key in dict`, agrupar datos |
| `collections.Counter` | `Counter`, `most_common`, `+`, `-`, `&` | Contar frecuencias, operaciones entre conteos |
| `collections.namedtuple` | `namedtuple("T", [campos])` | Tuplas con nombre, acceder por campo |
| `collections.ChainMap` | `ChainMap(dict1, dict2)` | Configuraciones con defaults + overrides |
| `itertools.chain` | `chain`, `chain.from_iterable` | Concatenar iterables sin copiar |
| `itertools.groupby` | `groupby(iterable, key)` | Agrupar elementos consecutivos |
| `itertools.product` | `product(A, B)` | Producto cartesiano, todos contra todos |
| `itertools.islice` | `islice(iter, start, stop, step)` | Rebanar iteradores (lazy) |
| `itertools.accumulate` | `accumulate(data, func)` | Acumulador running |
| `functools.reduce` | `reduce(func, iterable)` | Reducir iterable a un solo valor |
| `functools.partial` | `partial(func, arg=valor)` | Crear funciones pre-configuradas |
| `functools.lru_cache` | `@lru_cache(maxsize=N)` | Memoizacion automatica |
| `functools.singledispatch` | `@singledispatch` + `.register` | Dispatch por tipo de argumento |
| `functools.wraps` | `@wraps(func)` | Preservar metadatos de funciones decoradas |
| `pathlib.Path` | `Path`, `/`, `glob`, `rglob`, `read_text` | Manejo moderno de archivos y rutas |
| `logging` | `getLogger`, `basicConfig`, handlers | Logging profesional por niveles |
| `datetime` | `date`, `datetime`, `timedelta`, `timezone` | Fechas, horas, aritmetica temporal |
| `pprint` | `pprint`, `pformat` | Impresion legible de estructuras |
| `json` | `dumps`, `loads`, `JSONEncoder` | Serializacion/deserializacion JSON |
| `math` | `pi`, `sqrt`, `ceil`, `floor`, `dist` | Funciones matematicas |  
| `statistics` | `mean`, `median`, `stdev`, `quantiles` | Estadistica descriptiva |
| `timeit` | `timeit`, `Timer` | Medir rendimiento de codigo |

## 9. Ejercicios

### Ejercicio 1 (guiado): Analizar un log con Counter e itertools

Dado un archivo de logs de servidor, queremos:
1. Contar la frecuencia de cada tipo de error
2. Obtener las 3 lineas con mas errores
3. Encontrar el rango de tiempo con mas actividad

Completa el codigo siguiente.

In [ ]:
from collections import Counter, defaultdict
from itertools import groupby, islice

# Simular un archivo de logs del servidor
logs_simulados = """
2024-07-15 08:00:01 INFO  Conexion aceptada de 192.168.1.10
2024-07-15 08:00:02 WARNING Timeout de respuesta >5s
2024-07-15 08:00:03 ERROR  Base de datos no responde
2024-07-15 08:00:04 ERROR  Base de datos no responde
2024-07-15 08:00:05 INFO  Request completado en 200ms
2024-07-15 08:00:06 ERROR  Timeout de conexion a API externa
2024-07-15 08:00:07 WARNING Memoria al 90%
2024-07-15 08:00:08 INFO  Request completado en 150ms
2024-07-15 08:00:09 ERROR  Base de datos no responde
2024-07-15 08:00:10 CRITICAL Sistema fuera de disco
""".strip().split("\n")

# PASO 1: Contar cada tipo de error/nivel
niveles = Counter()
errores_por_minuto = defaultdict(int)

for linea in logs_simulados:
    partes = linea.split(maxsplit=4)
    nivel = partes[2]
    minuto = " ".join(partes[:2])[:16]  # YYYY-MM-DD HH:MM
    
    niveles[nivel] += 1
    if nivel in ("ERROR", "CRITICAL"):
        errores_por_minuto[minuto] += 1

print("Frecuencia por nivel:")
for nivel, count in niveles.most_common():
    barra = "#" * count
    print(f"  {nivel:8} | {barra} ({count})")

# PASO 2: Lineas con ERROR o CRITICAL
lineas_error = [l for l in logs_simulados if "ERROR" in l or "CRITICAL" in l]
print(f"\nLineas con error ({len(lineas_error)}):")
for linea in lineas_error:
    print(f"  {linea}")

# PASO 3: Minuto con mas errores
if errores_por_minuto:
    max_minuto = max(errores_por_minuto, key=errores_por_minuto.get)
    print(f"\nMinuto mas critico: {max_minuto} ({errores_por_minuto[max_minuto]} errores)")

### Ejercicio 2 (guiado): Pipeline con itertools

Usa `itertools` para crear un pipeline que:
1. Genere todas las combinaciones posibles de 3 productos (para comparar)
2. Filtre solo las donde los precios son distintos
3. Ordene por la diferencia de precios

In [ ]:
from itertools import combinations

productos = {
    "Laptop": 8999,
    "Tablet": 3499,
    "Monitor": 2999,
    "Teclado": 599,
    "Mouse": 299,
}

# Generar pares para comparar
comparaciones = []
for (nom_a, precio_a), (nom_b, precio_b) in combinations(productos.items(), 2):
    diff = abs(precio_a - precio_b)
    comparaciones.append((nom_a, nom_b, diff))

# Ordenar por diferencia (mayor a menor)
comparaciones.sort(key=lambda x: x[2], reverse=True)

print("Top 5 comparaciones por diferencia de precio:")
for a, b, diff in comparaciones[:5]:
    print(f"  {a:10} vs {b:10}: {diff:>7,} de diferencia")

### Ejercicio 3 (guiado): functools para configuraciones

Combina `partial`, `lru_cache` y `ChainMap` para crear un sistema de configuracion de un pipeline de datos.

In [ ]:
from functools import partial, lru_cache
from collections import ChainMap

# 1. Configuracion por defecto
defaults = {
    "batch_size": 1000,
    "timeout": 30,
    "retries": 3,
    "encoding": "utf-8",
    "verbose": False,
}

# 2. Overrides por ambiente
produccion = {
    "batch_size": 5000,
    "retries": 5,
    "verbose": False,
}

desarrollo = {
    "batch_size": 10,
    "verbose": True,
    "timeout": 5,
}

# 3. Configuracion final con ChainMap
config_prod = ChainMap(produccion, defaults)
config_dev = ChainMap(desarrollo, defaults)

print("Produccion:")
for k, v in config_prod.items():
    print(f"  {k}: {v}")

print("\nDesarrollo:")
for k, v in config_dev.items():
    print(f"  {k}: {v}")

# 4. Funcion de procesamiento con partial
def procesar_lote(datos, batch_size, timeout, verbose):
    if verbose:
        print(f"  Procesando lote de {len(datos)} items (timeout={timeout}s)")
    return sum(datos)

procesar_prod = partial(
    procesar_lote,
    batch_size=config_prod["batch_size"],
    timeout=config_prod["timeout"],
    verbose=config_prod["verbose"],
)

# 5. Resultado con cache
@lru_cache
def calcular_metricas(datos):
    return {"total": sum(datos), "media": sum(datos) / len(datos), "n": len(datos)}

resultado = procesar_prod(list(range(100)))
metricas = calcular_metricas(tuple(range(100)))  # tuple para hash
print(f"\nResultado: {resultado}")
print(f"Metricas: {metricas}")

### Ejercicio 4 (independiente): Analizar logs de una aplicacion web

Usando Counter e itertools, analiza los logs de una aplicacion web simulada:

1. Crea un diccionario con Counter que cuente cada tipo de status code (200, 301, 404, 500)
2. Usa `accumulate` para calcular el tiempo acumulado de respuesta
3. Con `takewhile` encuentra los primeros registros donde el tiempo de respuesta supera 200ms
4. Genera un reporte con los 5 endpoints mas visitados usando `most_common`

**Pista**: Usa los datos de ejemplo que te proporciono a continuacion como input.

In [ ]:
# Datos de ejemplo: (endpoint, status_code, tiempo_respuesta_ms)
accesos = [
    ("/api/users",       200, 45),
    ("/api/users",       200, 38),
    ("/api/products",    200, 120),
    ("/api/orders",      404, 12),
    ("/api/users",       301, 8),
    ("/api/products",    200, 95),
    ("/api/orders",      200, 210),  # > 200ms
    ("/api/users",       200, 55),
    ("/api/products",    200, 180),
    ("/api/orders",      500, 300),  # > 200ms
    ("/api/users",       200, 42),
    ("/api/products",    301, 5),
    ("/api/orders",      200, 150),
    ("/api/users",       200, 35),
    ("/api/products",    200, 88),
]

# Tu codigo aqui:
# 1. Contar status codes
# 2. Acumular tiempos con accumulate
# 3. takewhile para tiempos > 200ms
# 4. Endpoint mas visitado con most_common

print("Ejercicio 4 - Resuelve aqui")

## 10. Resumen

La libreria estandar de Python es una **herramienta mas poderosa** de lo que la mayoria de programadores sospecha. En esta sesion cubrimos:

| Concepto clave | Herramienta | Por que importa |
|----------------|-------------|------------------|
| **Contenedores especializados** | `collections` | `deque` para colas O(1), `Counter` para conteos, `defaultdict` para agrupar |
| **Iteradores avanzados** | `itertools` | Pipelines lazy: `chain`, `product`, `groupby`, `accumulate` |
| **Funciones de orden superior** | `functools` | `lru_cache` para memoizacion, `partial` para configurar, `singledispatch` por tipo |
| **Archivos modernos** | `pathlib` | Rutas como objetos con `/`, `glob`, `read_text`, timestamps |
| **Diagnostico profesional** | `logging` | Reemplaza `print()`: niveles, handlers, persistencia |
| **Fechas y horas** | `datetime` | `date`, `timedelta`, `timezone`, `strftime`/`strptime` |
| **Utilidades varias** | `pprint`, `json`, `math`, `statistics`, `timeit` | Legibilidad, serializacion, calculo, rendimiento |

### Regla de oro

> **Antes de `pip install`, revisa la stdlib.** La mayoria de problemas comunes (conteo, fechas, archivos, logs, configuracion) ya tienen solucion preinstalada. El programador avanzado domina su stdlib; el novato solo conoce `print`.

### Proximo paso

En la siguiente sesion veremos como integrar estas herramientas en **proyectos de ciencia de datos**: pipelines ETL con itertools, logging en modelos de ML, y gestion de fechas en series temporales.